Creating my weighted loss entropy

In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaForSequenceClassification, RobertaTokenizer, Trainer, TrainingArguments
from sklearn.metrics import f1_score

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # penalise missing a positive PCL case more heavily
        weights = torch.tensor([1.0, 8.5]).to(logits.device) 
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return {"f1": f1_score(labels, predictions, pos_label=1)}

training_args = TrainingArguments(
    output_dir="./BestModel",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)


trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset, 
    eval_dataset=dev_dataset,    
    compute_metrics=compute_metrics,
)

trainer.train()